<a href="https://colab.research.google.com/github/ParkHangah/AIFFEL_quest_eng/blob/master/Computer_Vision/CV03/D031~034_DLthon.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## 0.사전준비

### 0-1.구글 드라이브 연결

In [ ]:
from google.colab import drive
from IPython.display import clear_output, display
import ipywidgets as widgets
import os

def inf(msg, style, wdth): inf = widgets.Button(description=msg, disabled=True, button_style=style, layout=widgets.Layout(min_width=wdth));display(inf)

# 1. 구글 드라이브 마운트
print("Connecting...")
drive.mount('/content/gdrive')

# 2. 경로 설정 및 폴더 생성
base_path = os.path.join('/content/gdrive/MyDrive', '#Study/Aiffel/Work') # #Study/Aiffel/Work 부분에 드라이브 경로를 넣어주세요!
project_path = os.path.join(base_path, 'motorcycle')
kaggle_json_path = os.path.join(base_path, 'kaggle.json')
os.makedirs(project_path, exist_ok=True)

print(f"Selected Google Drive root path: {base_path}")
inf('\u2714 Done','success', '50px')

In [27]:
# 3. Kaggle API 설정 (가장 중요한 부분)
kaggle_config_dir = os.path.expanduser('~/.kaggle')
os.makedirs(kaggle_config_dir, exist_ok=True)
if os.path.exists(kaggle_json_path):
    # 드라이브의 파일을 시스템 설정 폴더로 복사
    !cp "{kaggle_json_path}" ~/.kaggle/
    !chmod 600 ~/.kaggle/kaggle.json
    print("✅ 구글 드라이브에서 Kaggle 인증 파일을 성공적으로 가져왔습니다.")
else:
    print(f"❌ '{kaggle_json_path}' 위치에 파일이 없습니다.")
    print("구글 드라이브의 해당 폴더에 kaggle.json을 먼저 업로드해주세요.")

✅ 구글 드라이브에서 Kaggle 인증 파일을 성공적으로 가져왔습니다.


In [29]:
# 4. 작업 디렉토리 이동 및 다운로드
# !cd 대신 %cd를 사용해야 이후의 명령어들이 이 경로에서 실행됩니다.
%cd "{project_path}"

/content/gdrive/MyDrive/#Study/Aiffel/Work/motorcycle


In [21]:
# 데이터셋 다운로드 및 압축 해제
print("📥 데이터셋 다운로드 및 압축 해제 중...")
!kaggle datasets download -d sadhliroomyprime/motorcycle-night-ride-semantic-segmentation
!unzip -qo motorcycle-night-ride-semantic-segmentation.zip -d ./data

print(f"\n현재 위치: {os.getcwd()}")
if os.path.exists('./data'):
    print("✨ 데이터 준비 완료! 폴더 내용:", os.listdir('./data'))

📥 데이터셋 다운로드 및 압축 해제 중...
Dataset URL: https://www.kaggle.com/datasets/sadhliroomyprime/motorcycle-night-ride-semantic-segmentation
License(s): Attribution 4.0 International (CC BY 4.0)
 97% 315M/325M [00:02<00:00, 113MB/s]
100% 325M/325M [00:02<00:00, 162MB/s]

현재 위치: /content/gdrive/MyDrive/#Study/Aiffel/Work/motorcycle
✨ 데이터 준비 완료! 폴더 내용: ['www.acmeai.tech ODataset 1 - Motorcycle Night Ride Dataset.pdf', 'www.acmeai.tech ODataset 1 - Motorcycle Night Ride Dataset']


In [13]:
# 데이터가 풀린 경로의 파일 목록 확인
data_list = os.listdir(os.path.join(mainpth, 'motorcycle'))
print("데이터 폴더 내용물:", data_list)

데이터 폴더 내용물: ['images', 'www.acmeai.tech ODataset 1 - Motorcycle Night Ride Dataset.pdf', 'COCO_motorcycle (pixel).json']


In [14]:
import os

# 1. 현재 코랩 메모리(로컬)에 있는지 확인
print("--- 코랩 로컬 경로 확인 ---")
!ls -d /content/data 2>/dev/null || echo "코랩 로컬에 data 폴더 없음"
!ls motorcycle-night-ride-semantic-segmentation.zip 2>/dev/null || echo "코랩 로컬에 zip 파일 없음"

# 2. 실제 드라이브 경로에 파일이 있는지 확인
print("\n--- 구글 드라이브 경로 확인 ---")
# 경로에 특수문자가 있으므로 반드시 따옴표로 감싸야 합니다.
if os.path.exists(project_path):
    print(f"폴더 존재함: {project_path}")
    print("내부 파일:", os.listdir(project_path))
else:
    print("폴더가 존재하지 않습니다. 경로를 다시 확인해주세요.")

--- 코랩 로컬 경로 확인 ---
코랩 로컬에 data 폴더 없음
코랩 로컬에 zip 파일 없음

--- 구글 드라이브 경로 확인 ---
폴더 존재함: /content/gdrive/MyDrive/#Study/Aiffel/Work/motorcycle
내부 파일: ['images', 'www.acmeai.tech ODataset 1 - Motorcycle Night Ride Dataset.pdf', 'COCO_motorcycle (pixel).json']


## 0-2.데이터 연결

In [22]:
import os
import json
from os.path import join
data_dir = os.path.join(project_path, 'data')
# hint : os.getenv를 사용하거나 직접 경로를 작성

json_data_path = os.path.join(data_dir, 'COCO_motorcycle (pixel).json')

print(json_data_path)

with open(json_data_path, 'r') as f:
    data = json.load(f)

# 제공되는 6개 클래스 정보 확인 [cite: 15-20]
categories = {cat['id']: cat['name'] for cat in data['categories']}
print(categories)
# 출력 예: {1: 'Undrivable', 2: 'Lanemark', 3: 'Road', 4: 'Movable', 5: 'My bike', 6: 'Rider'}

/content/gdrive/MyDrive/#Study/Aiffel/Work/motorcycle/data/COCO_motorcycle (pixel).json


FileNotFoundError: [Errno 2] No such file or directory: '/content/gdrive/MyDrive/#Study/Aiffel/Work/motorcycle/data/COCO_motorcycle (pixel).json'